# GitHub MCP + LangGraph — PR Reviewer

This is the **MCP version of the same PR-review problem** from the original notebook.

### Before
`LangGraph Agent → custom Python GitHub REST tools → GitHub API`

### Now
`LangGraph Agent → GitHub MCP tools → GitHub`

The LangGraph agent/tool loop remains. We are replacing the hand-written GitHub integration layer with the official GitHub MCP Server.

> **Safety:** this demo starts GitHub MCP in **read-only mode**, so the agent can inspect PRs but cannot merge, comment, edit, or delete anything.


## 0. Prerequisites

You need:

1. Python 3.10+
2. Docker Desktop installed and running
3. An OpenAI API key
4. A GitHub Personal Access Token that can read the target repository

Create a `.env` file beside this notebook:

```env
OPENAI_API_KEY=sk-...
GITHUB_PERSONAL_ACCESS_TOKEN=github_pat_...
```

For an older project that already uses `GITHUB_TOKEN`, this notebook supports that variable too.

Do **not** commit `.env` to GitHub.


In [ ]:
# Install the Python dependencies.
# Run this once, then restart the notebook kernel if Jupyter asks you to.

%pip install -U langchain langchain-openai langgraph langchain-mcp-adapters python-dotenv


In [1]:
# ============================================================
# 1. IMPORTS + ENVIRONMENT
# ============================================================

import os
import shutil
from typing import TypedDict, Annotated

from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

from langchain_mcp_adapters.client import MultiServerMCPClient

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Support both the official variable name and the name used
# in the original notebook.
GITHUB_TOKEN = (
    os.getenv("GITHUB_PERSONAL_ACCESS_TOKEN")
    or os.getenv("GITHUB_TOKEN")
)

assert OPENAI_API_KEY, "OPENAI_API_KEY is missing from .env"
assert GITHUB_TOKEN, (
    "Set GITHUB_PERSONAL_ACCESS_TOKEN (recommended) "
    "or GITHUB_TOKEN in your .env file."
)

assert shutil.which("docker"), (
    "Docker CLI was not found. Install/start Docker Desktop first."
)

print("✅ Environment variables loaded")
print("✅ Docker CLI found")


/Users/rahultiwari/Documents/02_Freelancing/coding_ninja_fresh/dummy-env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Environment variables loaded
✅ Docker CLI found


In [2]:
# ============================================================
# 2. DEMO CONFIGURATION
# ============================================================
# Change only these values when reviewing another PR.

OWNER = "rahul8879"
REPO = "e-comm-agentic-demo"
PR_NUMBER = 10

print(f"Repository : {OWNER}/{REPO}")
print(f"PR number  : #{PR_NUMBER}")


Repository : rahul8879/e-comm-agentic-demo
PR number  : #10


## 3. Connect to the official GitHub MCP Server

Python does **not** call GitHub REST endpoints directly anymore.

`MultiServerMCPClient` starts the official GitHub MCP Server as a Docker subprocess and discovers the tools that it exposes.

We intentionally enable only:

- `pull_requests`
- `repos`
- `actions`

and set `GITHUB_READ_ONLY=1`.


In [3]:
# ============================================================
# 3. GITHUB MCP CLIENT
# ============================================================

github_mcp = MultiServerMCPClient(
    {
        "github": {
            "transport": "stdio",
            "command": "docker",
            "args": [
                "run",
                "-i",
                "--rm",

                # Forward the PAT from the subprocess environment
                # into the Docker container.
                "-e",
                "GITHUB_PERSONAL_ACCESS_TOKEN",

                # Keep the tool surface focused for the LLM.
                "-e",
                "GITHUB_TOOLSETS=pull_requests,repos,actions",

                # IMPORTANT: no write operations in this demo.
                "-e",
                "GITHUB_READ_ONLY=1",

                "ghcr.io/github/github-mcp-server",
            ],
            "env": {
                "GITHUB_PERSONAL_ACCESS_TOKEN": GITHUB_TOKEN,
            },
        }
    }
)

# Jupyter supports top-level await.
tools = await github_mcp.get_tools()

print(f"✅ Loaded {len(tools)} GitHub MCP tools\n")

for tool in tools:
    print(f"• {tool.name}")


✅ Loaded 19 GitHub MCP tools

• actions_get
• actions_list
• get_commit
• get_file_contents
• get_job_logs
• get_latest_release
• get_release_by_tag
• get_tag
• list_branches
• list_commits
• list_pull_requests
• list_releases
• list_repository_collaborators
• list_tags
• pull_request_read
• search_code
• search_commits
• search_pull_requests
• search_repositories


In [4]:
# Optional: inspect the descriptions of PR-related tools.
# This is useful while teaching MCP tool discovery.

for tool in tools:
    if "pull" in tool.name.lower() or "pr" in tool.name.lower():
        print("=" * 90)
        print("TOOL:", tool.name)
        print(tool.description[:1500])
        print()


TOOL: list_pull_requests
List pull requests in a GitHub repository. If the user specifies an author, then DO NOT use this tool and use the search_pull_requests tool instead.

TOOL: pull_request_read
Get information on a specific pull request in GitHub repository.

TOOL: search_pull_requests
Search for pull requests in GitHub repositories using issues search syntax already scoped to is:pr



## 4. Build the LangGraph state

The original notebook stored `messages`, `pr_number`, and `verdict`.

Here we keep that idea, but also put `owner` and `repo` in state so the PR target belongs to the graph execution rather than relying on a global variable inside custom REST tools.


In [5]:
# ============================================================
# 4. LANGGRAPH STATE
# ============================================================

class PRReviewState(TypedDict):
    messages: Annotated[list, add_messages]
    owner: str
    repo: str
    pr_number: int
    verdict: str


In [6]:
# ============================================================
# 5. MODEL + MCP TOOLS
# ============================================================

llm = ChatOpenAI(
    model="gpt-4.1",
    temperature=0,
)

# These are NOT our own @tool functions.
# They were discovered from the GitHub MCP Server.
llm_with_tools = llm.bind_tools(tools)

print("✅ LLM bound to GitHub MCP tools")


✅ LLM bound to GitHub MCP tools


In [7]:
# ============================================================
# 6. SYSTEM PROMPT
# ============================================================

SYSTEM_PROMPT = """
You are CodeSentinel, an expert GitHub Pull Request reviewer.

You have GitHub tools supplied through MCP.

Your job is to investigate the requested pull request using tools,
then produce an evidence-based review.

When useful, inspect:
- pull request metadata and description
- changed files
- code diff
- commits
- reviews / review comments
- CI or check status

Rules:
1. Use GitHub MCP tools rather than guessing.
2. Do not invent repository, PR, CI, file, review, or code information.
3. This is a read-only review. Never attempt to modify GitHub.
4. Focus on correctness, bugs, security, maintainability, tests,
   backward compatibility, and operational risk.
5. If information is unavailable, explicitly say so.

Return the final answer in exactly this high-level structure:

## 🤖 CodeSentinel Review

### 📋 PR Summary
...

### ✅ What Looks Good
- ...

### ⚠️ Issues Found
- ...

### 🔧 Suggestions
- ...

### 🧪 CI / Status
- ...

### 📊 Verdict
APPROVED

or

NEEDS CHANGES

Then give a short reason for the verdict.
"""


## 7. Agent node + routing

This preserves the core pattern from the original notebook:

`START → agent → tools → agent → ... → END`

The LLM decides whether it needs another GitHub MCP tool call. We do **not** hard-code a sequence such as `details → files → commits → status`.


In [8]:
# ============================================================
# 7. AGENT NODE
# ============================================================

async def agent_node(state: PRReviewState):
    # Put the target PR into the system context for EVERY agent turn.
    # This avoids the global-PR-number problem from the REST version.
    task_context = f"""
Target repository: {state['owner']}/{state['repo']}
Target pull request: #{state['pr_number']}
"""

    messages = [
        SystemMessage(content=SYSTEM_PROMPT + "\n" + task_context),
        *state["messages"],
    ]

    response = await llm_with_tools.ainvoke(messages)

    return {
        "messages": [response],
    }


def should_continue(state: PRReviewState):
    """Route to ToolNode when the LLM requested tools; otherwise finish."""
    last_message = state["messages"][-1]

    if getattr(last_message, "tool_calls", None):
        return "tools"

    return END


In [9]:
# ============================================================
# 8. BUILD LANGGRAPH
# ============================================================

tool_node = ToolNode(tools)

builder = StateGraph(PRReviewState)

builder.add_node("agent", agent_node)
builder.add_node("tools", tool_node)

builder.add_edge(START, "agent")

builder.add_conditional_edges(
    "agent",
    should_continue,
    {
        "tools": "tools",
        END: END,
    },
)

# After MCP tools return evidence, go back to the LLM.
builder.add_edge("tools", "agent")

graph = builder.compile()

print("✅ LangGraph compiled")


✅ LangGraph compiled


In [10]:
# Optional graph visualization.
# If PNG rendering is unavailable in your environment,
# the Mermaid text below is still useful for teaching.

print(graph.get_graph().draw_mermaid())


---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	agent(agent)
	tools(tools)
	__end__([<p>__end__</p>]):::last
	__start__ --> agent;
	agent -.-> __end__;
	agent -.-> tools;
	tools --> agent;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 9. Review a real PR

Change `PR_NUMBER` in the configuration cell and rerun from there.

The agent is free to decide which GitHub MCP tools are necessary.


In [11]:
# ============================================================
# 9. RUN THE PR REVIEW
# ============================================================

initial_state: PRReviewState = {
    "messages": [
        HumanMessage(
            content=(
                f"Review PR #{PR_NUMBER} in {OWNER}/{REPO}. "
                "Investigate the PR with the available GitHub MCP tools "
                "and give me the final CodeSentinel review."
            )
        )
    ],
    "owner": OWNER,
    "repo": REPO,
    "pr_number": PR_NUMBER,
    "verdict": "",
}

result = await graph.ainvoke(
    initial_state,
    # Prevent accidental infinite agent/tool loops.
    config={"recursion_limit": 30},
)

print("✅ Review complete")


✅ Review complete


In [12]:
# ============================================================
# 10. FINAL REVIEW
# ============================================================

final_review = result["messages"][-1].content

print(final_review)


## 🤖 CodeSentinel Review

### 📋 PR Summary

- **Title:** added the dummy code
- **Author:** rahul8879
- **Branch:** feature/demo → main
- **Description:** "Please approve my code"
- **Files Changed:** 1 (`orders.py`)
- **Commits:** 2
- **Additions/Deletions:** +2/-1
- **PR URL:** [#10](https://github.com/rahul8879/e-comm-agentic-demo/pull/10)

### ✅ What Looks Good

- The PR is small and isolated to a single file (`orders.py`).
- The changes are clearly labeled as "dummy code" for testing purposes.

### ⚠️ Issues Found

- **Critical Security Vulnerabilities:**
  - **SQL Injection:** The code constructs SQL queries using string interpolation (f-strings) with unsanitized input, making it highly vulnerable to SQL injection attacks.
  - **Hardcoded Secret:** There is a hardcoded password in the code, which is a severe security risk.
  - **No Authentication Check:** The endpoint allows any user to access any order, violating basic access control principles.
- **Multiple Reviews Block the PR

In [13]:
# ============================================================
# 11. OPTIONAL — SEE THE FULL AGENT / TOOL TRACE
# ============================================================

for i, message in enumerate(result["messages"], start=1):
    print("\n" + "=" * 100)
    print(f"MESSAGE {i}: {type(message).__name__}")
    print("=" * 100)

    if getattr(message, "tool_calls", None):
        print("TOOL CALLS:")
        for call in message.tool_calls:
            print(call)

    content = getattr(message, "content", None)
    if content:
        print("\nCONTENT:")
        print(content)



MESSAGE 1: HumanMessage

CONTENT:
Review PR #10 in rahul8879/e-comm-agentic-demo. Investigate the PR with the available GitHub MCP tools and give me the final CodeSentinel review.

MESSAGE 2: AIMessage
TOOL CALLS:
{'name': 'pull_request_read', 'args': {'method': 'get', 'owner': 'rahul8879', 'repo': 'e-comm-agentic-demo', 'pullNumber': 10}, 'id': 'call_1owZiFvQJsCjeBUBq1tMU5ym', 'type': 'tool_call'}
{'name': 'pull_request_read', 'args': {'method': 'get_files', 'owner': 'rahul8879', 'repo': 'e-comm-agentic-demo', 'pullNumber': 10}, 'id': 'call_JWjLT9op8dBQYtweL9QWGx6e', 'type': 'tool_call'}
{'name': 'pull_request_read', 'args': {'method': 'get_commits', 'owner': 'rahul8879', 'repo': 'e-comm-agentic-demo', 'pullNumber': 10}, 'id': 'call_3i2Iya8AuxOgxidwddoIU0p9', 'type': 'tool_call'}
{'name': 'pull_request_read', 'args': {'method': 'get_reviews', 'owner': 'rahul8879', 'repo': 'e-comm-agentic-demo', 'pullNumber': 10}, 'id': 'call_9uRYVcx2Xuo9RXC2fOBlOlAw', 'type': 'tool_call'}
{'name': 'p

## Troubleshooting

### `docker` not found
Install Docker Desktop and make sure Docker is running.

### No MCP tools load
Check that Docker can pull/run:

```bash
docker pull ghcr.io/github/github-mcp-server
```

### GitHub authentication / 401 / 403
Check the PAT and its access to the target repository.

### Private repository
The PAT must have appropriate read access to that repository.

### OpenAI authentication error
Verify `OPENAI_API_KEY` in `.env`.

### Changed repository / PR
Edit only:

```python
OWNER = "..."
REPO = "..."
PR_NUMBER = ...
```

and rerun the notebook from the configuration cell onward.
